In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")
raw = f"/Volumes/{catalog}/bronze/raw"
for entity in ["orders", "customers", "products"]:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.{entity}")
    result = spark.sql(f"""
    COPY INTO {catalog}.bronze.{entity}
    FROM (
    SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
    FROM '{raw}/{entity}/'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true',
    'mergeSchema' = 'true')
    COPY_OPTIONS ('mergeSchema' = 'true')
    """)
    print(entity)
    display(result)

In [0]:
%sql
SELECT 'orders' AS t, COUNT(*) FROM shopstream_dev.bronze.orders
UNION ALL SELECT 'customers', COUNT(*) FROM shopstream_dev.bronze.customers
UNION ALL SELECT 'products', COUNT(*) FROM shopstream_dev.bronze.products